# Fábrica de Corpus CitizenTech (GS 2027)

Ferramenta que transforma documentos públicos de **saúde (dengue e vacinação no SUS)** em três corpus
prontos para o fine-tuning de um modelo de linguagem pequeno (SLM):

MEMBROS DO GRUPO

**Yuri Quirino - 568512**

**Leonardo Barros - 566788**

**Marcelo Augusto - 567176**

**Etapas:** ingestão → extração → limpeza → normalização → anonimização → segmentação → geração dos corpus → relatório de qualidade.

## 0. Instalação das bibliotecas

In [21]:
!pip -q install pypdf pdfplumber python-docx beautifulsoup4 lxml ftfy tiktoken spacy gradio google-genai requests
!python -m spacy download pt_core_news_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 1. Configuração
Tudo o que o grupo pode querer ajustar está nesta célula.

In [22]:
import os, re, json, math, time, random, hashlib, unicodedata
from pathlib import Path
from collections import Counter

import pandas as pd

TEMA = "Dengue e vacinação no SUS"

# Mensagem de sistema usada em todos os diálogos do corpus conversacional
SYSTEM_PROMPT = (
    "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, "
    "usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema "
    "ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a "
    "Unidade Básica de Saúde (UBS) ou o órgão responsável."
)

# Segmentação (em tokens)
MIN_TOKENS, MAX_TOKENS = 256, 512   # faixa exigida pelo enunciado
ALVO_TOKENS = 280                   # tamanho que tentamos atingir antes de fechar um trecho

# Geração
QA_POR_TRECHO = 3                   # pares de pergunta e resposta por trecho
TROCAS_POR_DIALOGO = 2              # trocas (usuário + atendente) por diálogo
LIMIAR_FIDELIDADE = 0.6             # fração mínima da resposta que precisa estar apoiada no trecho
MODELO_LLM = "gemini-flash-latest"  # troque por uma versão específica se preferir (veja ai.google.dev)
PAUSA_ENTRE_CHAMADAS = 7            # segundos entre chamadas (a camada gratuita aceita poucas por minuto)
ESPERA_LIMITE = 65                  # segundos de espera quando a API avisa que o limite por minuto estourou

# Revisão humana e reprodutibilidade
FRACAO_REVISAO = 0.20               # o enunciado pede revisão de pelo menos 20% dos exemplos gerados por IA
SEMENTE = 42

# Limpeza extra: tira créditos, ficha catalográfica e referências bibliográficas (não ajudam o cidadão)
REMOVER_CREDITOS_E_REFERENCIAS = True

PASTA_SAIDA = Path("saida")
PASTA_SAIDA.mkdir(exist_ok=True)
MINIMOS = {"texto": 50, "qa": 100, "chat": 30, "recusas": 10}

MOTIVO_CHAVE = ""   # preenchido quando a chave não é encontrada, para explicar o porquê


def ler_chave():
    """Procura a chave do Gemini nos Secrets do Colab (ou nas variáveis de ambiente)."""
    global MOTIVO_CHAVE
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    for nome in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        if userdata is not None:
            try:
                valor = userdata.get(nome)
                if valor:
                    MOTIVO_CHAVE = ""
                    return valor.strip()
            except Exception as erro:
                if "Access" in type(erro).__name__:
                    MOTIVO_CHAVE = f"o secret {nome} existe, mas o botão 'Notebook access' dele está desligado"
                elif not MOTIVO_CHAVE:
                    MOTIVO_CHAVE = "nenhum secret chamado GEMINI_API_KEY foi encontrado no painel Secrets"
        if os.environ.get(nome):
            return os.environ[nome].strip()
    return ""

### Contagem de tokens
Usamos o `tiktoken` (codificação `cl100k_base`) como referência. Se ele não estiver disponível,
caímos em uma estimativa simples para o notebook nunca parar.

In [23]:
try:
    import tiktoken
    _ENC = tiktoken.get_encoding("cl100k_base")
    TOKENIZADOR = "tiktoken (cl100k_base)"

    def contar_tokens(texto: str) -> int:
        return len(_ENC.encode(texto))
except Exception:
    TOKENIZADOR = "estimativa (1 token a cada ~4 caracteres de cada palavra)"

    def contar_tokens(texto: str) -> int:
        pecas = re.findall(r"\w+|[^\w\s]", texto)
        return sum(max(1, math.ceil(len(p) / 4)) if p[0].isalnum() else 1 for p in pecas)

print("Tokenizador em uso:", TOKENIZADOR)

Tokenizador em uso: tiktoken (cl100k_base)


## 2. Ingestão e extração de texto
Aceita **PDF, TXT, DOCX e HTML**, além de **texto colado**. Cada bloco extraído guarda a `fonte`
(nome do arquivo) e a `pagina`, para que todo trecho dos corpus possa ser rastreado até o documento de origem.

In [24]:
FORMATOS_ACEITOS = [".pdf", ".txt", ".docx", ".html", ".htm"]


def _decodificar(dados: bytes) -> str:
    """Lê bytes em qualquer codificação comum e devolve str (UTF-8 em memória)."""
    for cod in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            return dados.decode(cod)
        except UnicodeDecodeError:
            continue
    return dados.decode("utf-8", errors="replace")


def extrair_pdf(caminho):
    """Uma entrada por página. Tenta pdfplumber e, se falhar, pypdf."""
    try:
        import pdfplumber
        with pdfplumber.open(caminho) as pdf:
            return [(i, pag.extract_text() or "") for i, pag in enumerate(pdf.pages, 1)]
    except Exception:
        from pypdf import PdfReader
        return [(i, pag.extract_text() or "") for i, pag in enumerate(PdfReader(caminho).pages, 1)]


def extrair_docx(caminho):
    import docx
    doc = docx.Document(caminho)
    partes = [p.text for p in doc.paragraphs if p.text.strip()]
    for tabela in doc.tables:
        for linha in tabela.rows:
            celulas = [c.text.strip() for c in linha.cells if c.text.strip()]
            if celulas:
                partes.append(" | ".join(celulas))
    return [(1, "\n\n".join(partes))]


def extrair_html(bruto: str):
    from bs4 import BeautifulSoup, Comment
    try:
        sopa = BeautifulSoup(bruto, "lxml")
    except Exception:
        sopa = BeautifulSoup(bruto, "html.parser")
    # menus, scripts e rodapés do site não fazem parte do conteúdo
    for tag in sopa(["script", "style", "noscript", "nav", "header", "footer", "aside",
                     "form", "button", "svg", "iframe"]):
        tag.decompose()
    raiz = sopa.find("main") or sopa.find("article") or sopa.body or sopa
    for no in raiz.find_all(string=True):
        if isinstance(no, Comment):
            no.extract()
        else:
            no.replace_with(re.sub(r"\s+", " ", str(no)))
    for br in raiz.find_all("br"):
        br.replace_with("\n")
    for bloco in raiz.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "li", "tr", "div",
                                "section", "blockquote"]):
        bloco.insert_before("\n\n")
        bloco.insert_after("\n\n")
    return [(1, raiz.get_text())]


def extrair_documento(caminho, nome=None):
    """Devolve uma lista de blocos {'fonte', 'pagina', 'texto'} para um arquivo."""
    caminho = Path(caminho)
    nome = nome or caminho.name
    ext = caminho.suffix.lower()
    if ext == ".pdf":
        paginas = extrair_pdf(caminho)
    elif ext == ".docx":
        paginas = extrair_docx(caminho)
    elif ext in (".html", ".htm"):
        paginas = extrair_html(_decodificar(caminho.read_bytes()))
    elif ext == ".txt":
        paginas = [(1, _decodificar(caminho.read_bytes()))]
    else:
        raise ValueError(f"Formato não aceito: {ext}. Use {', '.join(FORMATOS_ACEITOS)}.")
    return [{"fonte": nome, "pagina": i, "texto": t} for i, t in paginas]


def ingerir(arquivos=None, texto_colado=""):
    """Etapas 1 e 2: lê todos os arquivos e o texto colado. Devolve {fonte: [blocos]} e os avisos."""
    documentos, avisos = {}, []
    for arq in arquivos or []:
        caminho = getattr(arq, "name", arq)          # o Gradio pode entregar caminho ou objeto
        nome = Path(caminho).name
        try:
            blocos = extrair_documento(caminho, nome)
            if not any(b["texto"].strip() for b in blocos):
                avisos.append(f"{nome}: nenhum texto extraído (PDF escaneado? seria preciso OCR).")
                continue
            documentos[nome] = blocos
        except Exception as erro:
            avisos.append(f"{nome}: falha na leitura ({erro}).")
    if texto_colado and texto_colado.strip():
        documentos["texto_colado"] = [{"fonte": "texto_colado", "pagina": 1, "texto": texto_colado}]
    return documentos, avisos


def tabela_origem(documentos):
    """Registro de origem: de qual arquivo e página veio cada bloco de texto."""
    linhas = []
    for fonte, blocos in documentos.items():
        for b in blocos:
            linhas.append({"fonte": fonte, "pagina": b["pagina"], "caracteres": len(b["texto"]),
                           "previa": re.sub(r"\s+", " ", b["texto"])[:120]})
    return pd.DataFrame(linhas, columns=["fonte", "pagina", "caracteres", "previa"])

## 3. Limpeza
Remove cabeçalhos e rodapés (linhas que se repetem no topo ou no fim de várias páginas), numeração de páginas,
linhas de sumário, URLs, caracteres estranhos e linhas repetidas. Também junta palavras quebradas por hífen no fim da linha
e letras iniciais que o PDF separa do resto da palavra ("A nexos" vira "Anexos").

Depois da anonimização, um segundo filtro tira parágrafos de **créditos, ficha catalográfica e referências bibliográficas**:
eles não trazem informação para o cidadão e concentram nomes de pessoas.

In [25]:
RE_URL = re.compile(r"(?:https?://|www\.)\S+", re.I)
RE_NUM_PAGINA = re.compile(
    r"^\s*(?:[-–—]\s*)?(?:p[áa]g(?:ina)?\.?\s*)?\d{1,4}(?:\s*(?:/|de)\s*\d{1,4})?(?:\s*[-–—])?\s*$", re.I)
RE_SUMARIO = re.compile(r"\.{4,}\s*\d{1,4}\s*$")
RE_HIFEN = re.compile(r"([A-Za-zÀ-ÖØ-öø-ÿ])-[ \t]*\n[ \t]*([a-zà-öø-ÿ])")
RE_MARCADOR = re.compile(r"^\s*[•●○■□▪▫◦‣▶►✓✔➢➤❖]\s*")
SIMBOLOS_MANTIDOS = {"°", "№"}


def _chave_linha(linha: str) -> str:
    """Normaliza a linha para comparar entre páginas (números viram #)."""
    return re.sub(r"\d+", "#", linha.strip().lower())


def detectar_cabecalhos_rodapes(paginas, n_bordas=3):
    """Linhas que aparecem no topo ou no fim de pelo menos 30% das páginas (mínimo de 3)."""
    if len(paginas) < 3:
        return set()
    contagem = Counter()
    for texto in paginas:
        linhas = [l for l in texto.splitlines() if l.strip()]
        bordas = linhas[:n_bordas] + linhas[-n_bordas:] if len(linhas) > 2 * n_bordas else linhas
        for chave in {_chave_linha(l) for l in bordas}:
            contagem[chave] += 1
    limite = max(3, math.ceil(0.3 * len(paginas)))
    return {c for c, n in contagem.items() if n >= limite and len(c) <= 120}


def _remover_caracteres_estranhos(texto, stats):
    saida = []
    for ch in texto:
        if ch in "\n\t":
            saida.append(ch)
            continue
        cat = unicodedata.category(ch)
        if cat == "Zs":                                   # qualquer espaço Unicode vira espaço comum
            saida.append(" ")
        elif cat in ("Cc", "Cf", "Co", "Cs", "Cn") or ch == "�":
            stats["caracteres_estranhos"] += 1            # controle, invisíveis, uso privado, "�"
        elif cat == "So" and ch not in SIMBOLOS_MANTIDOS:
            stats["caracteres_estranhos"] += 1            # ícones, dingbats, emojis
        else:
            saida.append(ch)
    return "".join(saida)


def _tirar_url(m):
    """Remove a URL, mas devolve a pontuação final para não quebrar a frase."""
    fim = re.search(r"[.,;:!?)\]]+$", m.group(0))
    return fim.group(0) if fim else ""


RE_LETRA_SOLTA = re.compile(r"^([ \t]*(?:- )?)([A-ZÀ-Ý]) ([a-zà-ÿ]+)", re.M)


def _vocabulario(paginas):
    """Contagem das palavras do documento, usada para consertar letras iniciais soltas."""
    todas = "\n".join(paginas)
    palavras = Counter(re.findall(r"[a-zà-ÿ]+", todas.lower()))
    soltas = Counter(m.group(3) for m in RE_LETRA_SOLTA.finditer(todas))
    return palavras, soltas


def _unir_letras_soltas(texto, vocabulario, stats, unidas):
    """Une "A nexos" -> "Anexos" só quando "anexos" existe no documento e "nexos" nunca aparece sozinho."""
    palavras, soltas = vocabulario

    def trocar(m):
        letra, resto = m.group(2), m.group(3)
        if palavras[(letra + resto).lower()] == 0:
            return m.group(0)
        if len(resto) == 1:
            if resto in "aeo":                            # "A a Z", "A e B": são palavras de verdade
                return m.group(0)
        elif palavras[resto] - soltas[resto] > 0:         # "A dengue": "dengue" aparece sozinha no texto
            return m.group(0)
        stats["letras_iniciais_unidas"] += 1
        if len(unidas) < 20:
            unidas.append(letra + resto)
        return m.group(1) + letra + resto

    return RE_LETRA_SOLTA.sub(trocar, texto)


def limpar_pagina(texto, cabecalhos, stats, vocabulario=None, unidas=None):
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")
    linhas = []
    for original in texto.split("\n"):
        linha = RE_MARCADOR.sub("- ", original)           # marcadores de lista viram "- "
        linha = _remover_caracteres_estranhos(linha, stats)
        if linha.strip():
            if _chave_linha(original) in cabecalhos:
                stats["cabecalhos_rodapes"] += 1
                continue
            if RE_NUM_PAGINA.match(linha):
                stats["numeros_de_pagina"] += 1
                continue
            if RE_SUMARIO.search(linha):
                stats["linhas_de_sumario"] += 1
                continue
        linhas.append(linha)
    texto = "\n".join(linhas)
    if vocabulario is not None:
        texto = _unir_letras_soltas(texto, vocabulario, stats, unidas if unidas is not None else [])

    texto, n = RE_URL.subn(_tirar_url, texto)
    stats["urls"] += n
    texto, n = RE_HIFEN.subn(r"\1\2", texto)
    stats["hifens_unidos"] += n

    limpas, anterior = [], None
    for linha in texto.split("\n"):
        nucleo = linha.strip()
        if nucleo and not re.search(r"[A-Za-zÀ-ÿ0-9]", nucleo):
            continue                                      # sobrou só pontuação (ex.: "()" de uma URL removida)
        if nucleo and nucleo == anterior:
            stats["linhas_repetidas"] += 1
            continue
        limpas.append(linha)
        anterior = nucleo or anterior
    return "\n".join(limpas)


def limpar_documento(blocos):
    """Etapa 3: limpa todas as páginas de um documento. Devolve (texto, estatísticas, letras unidas)."""
    stats, unidas = Counter(), []
    paginas = [b["texto"] for b in blocos]
    cabecalhos = detectar_cabecalhos_rodapes(paginas)
    vocabulario = _vocabulario(paginas)
    limpas = [limpar_pagina(p, cabecalhos, stats, vocabulario, unidas) for p in paginas]
    return "\n".join(limpas), stats, unidas


# ----- créditos, ficha catalográfica e referências (aplicado depois da anonimização) -----
RE_REFERENCIA = re.compile(
    r"Acesso em:|Disponível em|\bet al\.|\bISBN\b|\bISSN\b|\bv\. ?\d+, ?n\. ?\d+|^[A-ZÀ-Ý]{3,}, [A-ZÀ-Ý]\.", re.M)
RE_CREDITOS = re.compile(
    r"^(?:Tiragem|Ficha catalográfica|Catalogação na fonte|Títulos? para indexação|Produção editorial|"
    r"Projeto gráfico|Diagramação|Ilustraç(?:ão|ões)|Fotos?|Fotografias?|Capa|Revisão|Supervisão|Elaboração|"
    r"Colaboraç(?:ão|ões)|Colaboradores|Organização|Coordenação|Edição|Editoração|Normalização|Impressão|"
    r"Equipe técnica|Autor(?:es|ia)?|Apoio|Agradecimentos)\b[^.!?\n]{0,60}:"
    r"|Impresso no Brasil|Printed in Brazil|Esplanada dos Ministérios|\bCEP:? ?\d{5}|\bFax:|Endereço eletrônico|"
    r"Todos os direitos reservados|permitida a reprodução|©|\bCDU\b|\bCDD\b|venda proibida", re.I | re.M)


def remover_creditos_e_referencias(texto):
    """Devolve (texto sem parágrafos editoriais, lista dos parágrafos removidos)."""
    mantidos, removidos = [], []
    for paragrafo in texto.split("\n\n"):
        editorial = (RE_REFERENCIA.search(paragrafo) or RE_CREDITOS.search(paragrafo)
                     or paragrafo.count("[NOME]") >= 3)     # listas de autores e colaboradores
        (removidos if editorial else mantidos).append(paragrafo)
    return "\n\n".join(mantidos), removidos

## 4. Normalização
Padroniza codificação (UTF-8, forma NFC), espaços, quebras de linha e aspas.
**Acentos, pontuação e maiúsculas são mantidos.**

In [26]:
ASPAS = {"“": '"', "”": '"', "„": '"', "«": '"', "»": '"', "‟": '"',
         "‘": "'", "’": "'", "‚": "'", "‛": "'"}
LIGADURAS = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl"}
PALAVRAS_DE_LIGACAO = {"a", "o", "as", "os", "de", "da", "do", "das", "dos", "e", "em", "no", "na",
                       "nos", "nas", "para", "por", "com", "que", "um", "uma", "ou", "ao", "à", "se"}


def _parece_titulo(linha, proxima):
    """Título: linha curta, sem pontuação final, seguida de linha que começa com maiúscula."""
    if not linha or linha[-1] in ".,;:!?" or len(linha) > 60 or linha.startswith("- "):
        return False
    letras = [c for c in linha if c.isalpha()]
    if len(letras) >= 3 and all(c.isupper() for c in letras):
        return True
    ultima = linha.split()[-1].lower()
    return len(linha.split()) <= 8 and proxima[:1].isupper() and ultima not in PALAVRAS_DE_LIGACAO


def _juntar_linhas(texto):
    """Desfaz as quebras de linha no meio dos parágrafos (comuns em PDF)."""
    paragrafos = []
    for bloco in re.split(r"\n[ \t]*\n", texto):
        linhas = [l.strip() for l in bloco.split("\n") if l.strip()]
        if not linhas:
            continue
        atual = linhas[0]
        for linha in linhas[1:]:
            fim_de_frase = atual[-1] in ".!?:;" and (linha[0].isupper() or linha[0].isdigit()
                                                     or linha[0] in "\"'([")
            if linha.startswith("- ") or fim_de_frase or _parece_titulo(atual, linha):
                paragrafos.append(atual)
                atual = linha
            else:
                atual += " " + linha
        paragrafos.append(atual)
    return "\n\n".join(paragrafos)


def normalizar(texto: str) -> str:
    """Etapa 4: devolve o texto padronizado, com um parágrafo por bloco separado por linha em branco."""
    try:
        import ftfy
        texto = ftfy.fix_encoding(texto)                 # conserta "vacinaÃ§Ã£o" -> "vacinação"
    except Exception:
        pass
    texto = unicodedata.normalize("NFC", texto)          # NFC mantém os acentos (não usamos NFKD/ASCII)
    for origem, destino in {**ASPAS, **LIGADURAS}.items():
        texto = texto.replace(origem, destino)
    texto = texto.replace("\r\n", "\n").replace("\r", "\n").replace("\t", " ")
    texto = _juntar_linhas(texto)
    texto = re.sub(r" {2,}", " ", texto)
    texto = re.sub(r" +([,.;:!?)\]])", r"\1", texto)     # sem espaço antes da pontuação
    texto = re.sub(r"([(\[]) +", r"\1", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

## 5. Anonimização (LGPD)
Mascara **CPF, e-mail, telefone e nomes de pessoas** antes de gerar qualquer corpus.

- CPF, e-mail e telefone: expressões regulares.
- Nomes: reconhecimento de entidades do spaCy (`pt_core_news_sm`) + regras (títulos como "Dra." e primeiros nomes comuns).
- Telefones de utilidade pública (0800, 136, 192) **não** são dados pessoais e são úteis ao cidadão, por isso ficam no texto.
- Nomes de doenças, vacinas e órgãos são protegidos por uma lista para não serem confundidos com pessoas.

In [27]:
try:
    import spacy
    NLP = spacy.load("pt_core_news_sm")
except Exception as _erro:
    NLP = None
    print("spaCy indisponível; nomes serão detectados apenas por regras.", _erro)

RE_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
RE_CPF = re.compile(r"(?<![\d.])\d{3}\.?\d{3}\.?\d{3}\s?-?\s?\d{2}(?![\d])")
RE_TELEFONE = re.compile(
    r"(?P<publico>\b0800[\s.-]?\d{2,3}[\s.-]?\d{3,4}\b)"                       # 0800: mantido
    r"|(?P<ddd>(?:\+?55\s?)?\(?\b\d{2}\)?\s?9?\s?\d{4}[-.\s]?\d{4}\b)"         # com DDD
    r"|(?P<local>\b9?\d{4}-\d{4}\b)")                                          # sem DDD
RE_ANOS = re.compile(r"^(?:19|20)\d{2}-(?:19|20)\d{2}$")                       # 2023-2024 não é telefone

_PARTE_NOME = r"[A-ZÀ-Ý][a-zà-ÿ]+"
_SOBRENOMES = rf"(?:\s+(?:d[aeo]s?\s+|e\s+)?{_PARTE_NOME}){{1,5}}"
RE_NOME_TITULO = re.compile(
    rf"\b((?:Dr|Dra|Sr|Sra|Srta|Prof|Profa|Enf|Enfa)\.|Doutora?|Senhora?|Professora?|Enfermeir[oa])"
    rf"\s+({_PARTE_NOME}(?:\s+(?:d[aeo]s?\s+)?{_PARTE_NOME}){{0,4}})")
NOMES_COMUNS = ("Maria|José|João|Ana|Antônio|Antonio|Francisco|Carlos|Paulo|Pedro|Lucas|Luiz|Luís|Marcos|"
                "Gabriel|Rafael|Daniel|Marcelo|Bruno|Eduardo|Felipe|Rodrigo|Fernando|Gustavo|Ricardo|"
                "Juliana|Fernanda|Patrícia|Patricia|Aline|Camila|Amanda|Bruna|Jéssica|Letícia|Júlia|"
                "Mariana|Adriana|Sandra|Márcia|Marcia|Cláudia|Claudia|Luciana|Vanessa|Beatriz|Larissa|"
                "Roberto|Sérgio|Sergio|Jorge|André|Andre|Mateus|Matheus|Thiago|Tiago|Diego|Leonardo|"
                "Renata|Simone|Cristina|Débora|Carla|Raquel|Helena|Alice|Laura|Sofia|Miguel|Arthur|Heitor")
RE_NOME_COMUM = re.compile(rf"(?<!São )(?<!Santa )(?<!Santo )\b(?:{NOMES_COMUNS}){_SOBRENOMES}")
# em listas ("[NOME], Fulana de Tal e [NOME]"), o vizinho de um nome mascarado também é um nome
_SEQUENCIA = rf"{_PARTE_NOME}(?:\s+(?:d[aeo]s?\s+)?{_PARTE_NOME}){{1,4}}"
RE_VIZINHO_DEPOIS = re.compile(rf"(\[NOME\](?:,| e) )({_SEQUENCIA})(?=[,.;]| e |$)")
RE_VIZINHO_ANTES = re.compile(rf"()({_SEQUENCIA})(?=(?:,| e) \[NOME\])")

# termos do tema que o reconhecedor de entidades costuma confundir com nomes de pessoas
VOCABULARIO_PROTEGIDO = {
    "aedes", "aegypti", "albopictus", "zika", "chikungunya", "dengue", "sus", "saúde", "ministério",
    "secretaria", "brasil", "vacina", "vacinação", "covid", "bcg", "hpv", "influenza", "febre", "amarela",
    "ubs", "upa", "samu", "unidade", "básica", "hospital", "governo", "estado", "município", "prefeitura",
    "programa", "nacional", "calendário", "hepatite", "sarampo", "rubéola", "caxumba", "tétano", "difteria",
    "poliomielite", "rotavírus", "meningite", "pneumonia", "gripe", "anvisa", "fiocruz", "butantan", "opas",
    "oms", "disque", "agente", "comunitário", "atenção", "vigilância", "tríplice", "viral", "pentavalente",
    "varicela", "coqueluche", "criança", "adolescente", "adulto", "idoso", "gestante", "cartilha", "manual",
    "são", "santo", "santa", "nossa", "senhora",          # "espada de São Jorge", "Santa Casa"
}
_PARTICULAS = {"de", "da", "do", "das", "dos", "e"}
# nomes próprios que aparecem em instituições e leis (não identificam um cidadão)
NOMES_INSTITUCIONAIS = {"oswaldo cruz", "adolfo lutz", "emílio ribas", "evandro chagas", "carlos chagas",
                        "vital brazil", "maria da penha"}


def _protegido(nome: str) -> bool:
    return nome.strip().lower() in NOMES_INSTITUCIONAIS


def _eh_nome_de_pessoa(texto: str) -> bool:
    palavras = texto.split()
    if _protegido(texto):
        return False
    if not (2 <= len(palavras) <= 7) or "\n" in texto or any(c.isdigit() for c in texto):
        return False
    for p in palavras:
        nucleo = p.strip(".,;:()")
        if nucleo.lower() in _PARTICULAS:
            continue
        if nucleo.lower() in VOCABULARIO_PROTEGIDO or not nucleo[:1].isupper() or nucleo.isupper():
            return False
    return True


def _mascarar_nomes(paragrafo, contagem):
    def trocar(m, prefixo=""):
        nome = m.group(2) if prefixo else m.group(0)
        if _protegido(nome):
            return m.group(0)
        contagem["nomes"] += 1
        return (m.group(1) + " " if prefixo else "") + "[NOME]"

    paragrafo = RE_NOME_TITULO.sub(lambda m: trocar(m, "titulo"), paragrafo)
    paragrafo = RE_NOME_COMUM.sub(trocar, paragrafo)
    if NLP is not None and re.search(r"[A-ZÀ-Ý][a-zà-ÿ]+\s+(?:d[aeo]s?\s+)?[A-ZÀ-Ý][a-zà-ÿ]+", paragrafo):
        doc = NLP(paragrafo)
        for ent in reversed(doc.ents):                    # de trás para frente para não deslocar os índices
            if ent.label_ == "PER" and _eh_nome_de_pessoa(ent.text):
                paragrafo = paragrafo[:ent.start_char] + "[NOME]" + paragrafo[ent.end_char:]
                contagem["nomes"] += 1

    def vizinho(m):
        if not _eh_nome_de_pessoa(m.group(2)):
            return m.group(0)
        contagem["nomes"] += 1
        return m.group(1) + "[NOME]"

    for _ in range(5):                                    # repete até a lista inteira estar mascarada
        novo = RE_VIZINHO_ANTES.sub(vizinho, RE_VIZINHO_DEPOIS.sub(vizinho, paragrafo))
        if novo == paragrafo:
            break
        paragrafo = novo
    return paragrafo


def _mascarar_telefone(m, contagem):
    if m.group("publico") or RE_ANOS.match(m.group(0)):
        return m.group(0)
    contagem["telefones"] += 1
    return "[TELEFONE]"


def _janela(antes, depois, largura=320):
    """Recorta os dois textos em volta da primeira diferença, para o exemplo antes/depois ficar legível."""
    i = next((k for k, (a, b) in enumerate(zip(antes, depois)) if a != b), 0)
    inicio = max(0, i - 90)
    return {"antes": antes[inicio:inicio + largura], "depois": depois[inicio:inicio + largura]}


def anonimizar(texto: str):
    """Etapa 5: devolve (texto mascarado, contagem por tipo, exemplos antes/depois)."""
    contagem, exemplos, saida = Counter(), [], []
    for paragrafo in texto.split("\n\n"):
        novo, n = RE_EMAIL.subn("[EMAIL]", paragrafo)
        contagem["emails"] += n
        novo, n = RE_CPF.subn("[CPF]", novo)
        contagem["cpfs"] += n
        novo = RE_TELEFONE.sub(lambda m: _mascarar_telefone(m, contagem), novo)
        novo = _mascarar_nomes(novo, contagem)
        if novo != paragrafo and len(exemplos) < 3:
            exemplos.append(_janela(paragrafo, novo))
        saida.append(novo)
    return "\n\n".join(saida), contagem, exemplos


# Texto fictício para demonstrar a anonimização (nenhum dado real)
TEXTO_DEMO_ANONIMIZACAO = (
    "Relato de atendimento (dados fictícios). A paciente Maria Aparecida de Souza, CPF 123.456.789-09, "
    "procurou a UBS com febre e dor no corpo. Ela foi atendida pela Dra. Helena Martins e orientada a "
    "beber bastante água. O retorno foi marcado pelo telefone (11) 91234-5678 e a confirmação enviada "
    "para maria.souza@exemplo.com. Em caso de dúvida, a população pode ligar para o Disque Saúde 136."
)

_demo, _cont, _ = anonimizar(TEXTO_DEMO_ANONIMIZACAO)
print("ANTES :", TEXTO_DEMO_ANONIMIZACAO)
print("DEPOIS:", _demo)
print("Mascarados:", dict(_cont))

ANTES : Relato de atendimento (dados fictícios). A paciente Maria Aparecida de Souza, CPF 123.456.789-09, procurou a UBS com febre e dor no corpo. Ela foi atendida pela Dra. Helena Martins e orientada a beber bastante água. O retorno foi marcado pelo telefone (11) 91234-5678 e a confirmação enviada para maria.souza@exemplo.com. Em caso de dúvida, a população pode ligar para o Disque Saúde 136.
DEPOIS: Relato de atendimento (dados fictícios). A paciente [NOME], CPF [CPF], procurou a UBS com febre e dor no corpo. Ela foi atendida pela Dra. [NOME] e orientada a beber bastante água. O retorno foi marcado pelo telefone [TELEFONE] e a confirmação enviada para [EMAIL]. Em caso de dúvida, a população pode ligar para o Disque Saúde 136.
Mascarados: {'emails': 1, 'cpfs': 1, 'telefones': 1, 'nomes': 2}


## 6. Segmentação
Divide o texto em trechos de **256 a 512 tokens**, sempre fechando no fim de uma frase ou parágrafo.
Se o fim do documento deixar uma sobra curta, ela é somada aos trechos anteriores e o conjunto é redividido
em partes iguais, de novo nos fins de frase.

In [28]:
ABREVIACOES = {"dr", "dra", "sr", "sra", "srta", "prof", "profa", "enf", "art", "arts", "nº", "n", "p",
               "pág", "ex", "av", "tel", "obs", "inc", "fl", "fls", "cap", "vol", "ed", "min", "máx"}
_FOLGA = 10                      # margem para diferenças de contagem ao juntar frases
_MAX_UNIDADE = 110               # frases maiores que isso são partidas (listas longas sem ponto final)


def dividir_frases(paragrafo: str):
    partes = re.split(r"(?<=[.!?])\s+(?=[\"'(\[]?[A-ZÀ-Ý0-9\-])", paragrafo.strip())
    frases = []
    for parte in partes:
        m = re.search(r"(?:^|\s)([\wº]{1,5})\.$", frases[-1]) if frases else None
        if m and m.group(1).lower() in ABREVIACOES:
            frases[-1] += " " + parte                     # "Dr." não encerra a frase
        elif parte.strip():
            frases.append(parte.strip())
    return frases


def _partir_frase_longa(frase):
    if contar_tokens(frase) <= _MAX_UNIDADE:
        return [frase]
    pedacos, atual, total = [], [], 0
    for palavra in frase.split():
        n = contar_tokens(" " + palavra)
        if atual and total + n > _MAX_UNIDADE:
            pedacos.append(" ".join(atual))
            atual, total = [], 0
        atual.append(palavra)
        total += n
    if atual:
        pedacos.append(" ".join(atual))
    return pedacos


def _montar(unidades):
    return "".join(u["t"] + ("\n" if u["fim"] else " ") for u in unidades).strip()


def _total(unidades):
    return contar_tokens(_montar(unidades)) if unidades else 0


def _repartir(unidades, n):
    """Divide as unidades em n partes, cortando nos fins de frase mais próximos dos pontos ideais."""
    if n <= 1:
        return [unidades]
    acumulado, soma = [], 0
    for u in unidades:
        soma += u["n"]
        acumulado.append(soma)
    cortes, anterior = [], 0
    for k in range(1, n):
        candidatos = range(anterior + 1, len(unidades) - (n - k) + 1)
        if not candidatos:
            return None
        corte = min(candidatos, key=lambda j: abs(acumulado[j - 1] - k * soma / n))
        cortes.append(corte)
        anterior = corte
    limites = [0] + cortes + [len(unidades)]
    return [unidades[a:b] for a, b in zip(limites, limites[1:])]


def _encaixar_sobra(grupos, sobra):
    """Soma a sobra aos últimos trechos e redivide tudo dentro da faixa de tokens."""
    centro = (MIN_TOKENS + MAX_TOKENS) / 2
    for recuo in sorted({1, 2, 3, len(grupos)}):
        if recuo > len(grupos):
            continue
        junto = [u for g in grupos[-recuo:] for u in g] + sobra
        total = _total(junto)
        opcoes = [n for n in range(1, recuo + 3) if MIN_TOKENS <= total / n <= MAX_TOKENS]
        for n in sorted(opcoes, key=lambda n: abs(total / n - centro)):
            partes = _repartir(junto, n)
            if partes and all(MIN_TOKENS <= _total(p) <= MAX_TOKENS for p in partes):
                return grupos[:-recuo] + partes
    return grupos + [sobra]                               # não há fim de frase que permita ficar na faixa


def segmentar(texto: str, fonte: str):
    """Etapa 6: devolve uma lista de {'text', 'fonte', '_tokens'}."""
    unidades = []
    for paragrafo in texto.split("\n\n"):
        pedacos = [p for f in dividir_frases(paragrafo) for p in _partir_frase_longa(f)]
        for i, p in enumerate(pedacos):
            unidades.append({"t": p, "fim": i == len(pedacos) - 1, "n": contar_tokens(p) + 1})

    grupos, atual, total = [], [], 0
    piso = MIN_TOKENS + _FOLGA
    for u in unidades:
        if atual and total >= piso and total + u["n"] > max(ALVO_TOKENS, piso) + 60:
            grupos.append(atual)
            atual, total = [], 0
        atual.append(u)
        total += u["n"]
        if u["fim"] and total >= max(ALVO_TOKENS, piso):   # preferimos fechar no fim do parágrafo
            grupos.append(atual)
            atual, total = [], 0

    if atual:                                              # sobra do fim do documento
        if total >= piso or not grupos:
            grupos.append(atual)
        else:
            grupos = _encaixar_sobra(grupos, atual)

    trechos = []
    for g in grupos:
        t = _montar(g)
        if t:
            trechos.append({"text": t, "fonte": fonte, "_tokens": contar_tokens(t),
                            "_doc_curto": len(grupos) == 1})   # documento inteiro menor que um trecho
    return trechos

## 7. Geração dos três corpus

- **Texto puro:** os próprios trechos da segmentação.
- **Perguntas e respostas** e **conversacional:** gerados com um modelo de linguagem (Gemini) quando há chave de API,
  ou por regras quando não há. Toda resposta gerada por IA passa por um **filtro de fidelidade**: as palavras e,
  principalmente, os números da resposta precisam estar no trecho de origem; caso contrário o exemplo é descartado.
- **Limites do assistente:** exemplos escritos à mão em que o atendente recusa perguntas fora do tema
  ou encaminha para um profissional.

In [29]:
PROMPT_GERACAO = """Você está criando dados de treino para um assistente de saúde pública do SUS que atende \
cidadãos com pouca familiaridade com termos técnicos. Tema do assistente: {tema}.

Use SOMENTE as informações do TRECHO abaixo. Não invente nada: números, datas, idades, nomes de vacinas, \
doses e orientações só podem aparecer se estiverem escritos no trecho.

Gere:
1. {n_qa} pares de pergunta e resposta. As perguntas devem soar como dúvidas reais de um cidadão, em linguagem \
simples. As respostas devem ser claras, curtas (1 a 3 frases), sem termos técnicos desnecessários e totalmente \
apoiadas no trecho.
2. Um diálogo com {n_trocas} trocas entre o cidadão e o atendente, em que cada nova pergunta dá continuidade \
à anterior.

Se o trecho não trouxer informação útil para o cidadão (sumário, créditos, ficha técnica, referências), \
devolva as duas listas vazias.

Responda apenas com JSON neste formato:
{{"qa": [{{"pergunta": "...", "resposta": "..."}}], "dialogo": [{{"usuario": "...", "assistente": "..."}}]}}

TRECHO (fonte: {fonte}):
<<<
{trecho}
>>>"""


def criar_cliente_llm(api_key: str):
    if not api_key:
        return None
    try:
        from google import genai
        return genai.Client(api_key=api_key)
    except Exception as erro:
        print("Não foi possível iniciar o cliente do Gemini:", erro)
        return None


ULTIMO_ERRO_LLM = ""


def chamar_llm(cliente, prompt, tentativas=4):
    """Devolve o texto da resposta ou None se todas as tentativas falharem.
    Erro 429 (limite por minuto) ou 503 (modelo sobrecarregado): espera e tenta de novo."""
    global ULTIMO_ERRO_LLM
    from google.genai import types
    config = types.GenerateContentConfig(temperature=0.4, response_mime_type="application/json")
    for tentativa in range(tentativas):
        try:
            resposta = cliente.models.generate_content(model=MODELO_LLM, contents=prompt, config=config)
            return resposta.text
        except Exception as erro:
            ULTIMO_ERRO_LLM = str(erro)[:300]
            texto = str(erro)
            if "PerDay" in texto or "per day" in texto.lower():
                print("  Cota DIÁRIA da API esgotada. Tente amanhã ou use outra chave.")
                return None
            limite = any(c in texto for c in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "overloaded"))
            espera = ESPERA_LIMITE if limite else PAUSA_ENTRE_CHAMADAS * (2 ** tentativa)
            print(f"  LLM falhou (tentativa {tentativa + 1}/{tentativas}): {texto[:120]} | aguardando {espera}s")
            time.sleep(espera)
    return None


def testar_llm(api_key):
    """Faz uma chamada curta para confirmar que a chave e o modelo funcionam. Se o modelo configurado
    não existir para a chave, procura outro modelo 'flash' disponível e passa a usá-lo."""
    global MODELO_LLM
    cliente = criar_cliente_llm(api_key)
    if cliente is None:
        print("Sem chave do Gemini" + (f" ({MOTIVO_CHAVE})" if MOTIVO_CHAVE else "") +
              ". Perguntas e diálogos serão gerados por regras.")
        return False
    candidatos = [MODELO_LLM]
    try:
        nomes = [m.name.split("/")[-1] for m in cliente.models.list()]
        candidatos += [n for n in nomes if "flash" in n and n != MODELO_LLM and not any(
            x in n for x in ("image", "tts", "audio", "live", "lite", "embedding", "exp"))]
    except Exception as erro:
        print("Não consegui listar os modelos:", str(erro)[:120])
    for nome in candidatos[:6]:
        try:
            r = cliente.models.generate_content(model=nome, contents="Responda apenas com a palavra: ok")
            MODELO_LLM = nome
            print(f"Gemini funcionando com o modelo '{nome}'. Resposta de teste: {(r.text or '').strip()[:20]}")
            return True
        except Exception as erro:
            print(f"Falha com o modelo '{nome}': {str(erro)[:160]}")
    return False


def _ler_json(bruto):
    if not bruto:
        return None
    bruto = re.sub(r"^```(?:json)?|```$", "", bruto.strip(), flags=re.M).strip()
    try:
        return json.loads(bruto)
    except Exception:
        m = re.search(r"\{.*\}", bruto, re.S)
        try:
            return json.loads(m.group(0)) if m else None
        except Exception:
            return None


def fidelidade(resposta: str, trecho: str) -> float:
    """Fração das palavras da resposta que estão no trecho. Zero se houver número que não está no trecho.
    A comparação pelos 5 primeiros caracteres serve só para este filtro (aceitar 'vacinas'/'vacinar');
    o texto dos corpus nunca é alterado por stemming."""
    palavras = re.findall(r"[a-zà-ÿ]{4,}", resposta.lower())
    if not palavras:
        return 0.0
    base = {p[:5] for p in re.findall(r"[a-zà-ÿ]{4,}", trecho.lower())}
    numeros_ok = all(n in trecho for n in re.findall(r"\d+(?:[.,]\d+)?", resposta))
    return sum(p[:5] in base for p in palavras) / len(palavras) if numeros_ok else 0.0


# ---------- geração por regras (sempre fiel: as respostas são frases do próprio documento) ----------
_ARTIGOS = {"o", "a", "os", "as", "um", "uma", "uns", "umas"}
_VAGOS = {"esse", "essa", "esses", "essas", "este", "esta", "estes", "estas", "aquele", "aquela", "todo", "toda",
          "todos", "todas", "outro", "outra", "outros", "outras", "cada", "algum", "alguma", "alguns", "algumas",
          "qualquer", "seu", "sua", "seus", "suas", "mesmo", "mesma"}


def _frases_do_trecho(trecho):
    return [f for linha in trecho.split("\n") for f in dividir_frases(linha)]


def _assunto(frase):
    """Escolhe um sintagma nominal da frase para montar a pergunta."""
    if NLP is not None:
        try:
            candidatos = [c.text for c in NLP(frase).noun_chunks
                          if len(c) <= 6 and "[" not in c.text and c[0].lower_ not in _VAGOS
                          and sum(t.pos_ != "DET" for t in c) >= 2      # "a água" é vago demais
                          and c.root.pos_ in ("NOUN", "PROPN") and c[0].pos_ in ("DET", "NOUN", "PROPN", "ADJ", "NUM")]
            if candidatos:
                return max(candidatos, key=len)
        except Exception:
            pass
    m = re.search(r"\b((?:[oa]s?)\s+[a-zà-ÿ]{4,}\s+d[aeo]s?\s+[a-zà-ÿ]{4,})", frase, re.I)
    return m.group(1) if m else None


def qa_por_regras(trecho: str, n: int):
    frases = _frases_do_trecho(trecho)
    pares, vistos = [], set()

    def adicionar(pergunta, resposta):
        chave = pergunta.lower()
        resposta = re.sub(r"^- ", "", resposta)
        if not (resposta[:1].isupper() or resposta[:1].isdigit()):
            return                                         # pedaço de frase: não serve como resposta
        if chave not in vistos and 5 <= len(resposta.split()) <= 90:
            vistos.add(chave)
            pares.append((pergunta, resposta))

    for i, f in enumerate(frases[:-1]):                    # 1) título em forma de pergunta + resposta
        if f.endswith("?") and 10 <= len(f) <= 160:
            resposta = [g for g in frases[i + 1:i + 3] if not g.endswith("?")]
            if resposta:
                adicionar(f, " ".join(resposta))
    for f in frases:                                       # 2) definições: "A dengue é ..."
        m = re.match(r"^(A|O|As|Os)\s+([^,.;:?!]{3,60}?)\s+(é|são)\s+(?:uma?|[oa]s?|aquel[ea]s?)\s", f)
        if m and len(f.split()) >= 8 and not f.endswith("?"):
            adicionar(f"O que {m.group(3)} {m.group(1).lower()} {m.group(2)}?", f)
    for f in frases:                                       # 3) "O que o material informa sobre X?"
        if len(pares) >= n:
            break
        if 8 <= len(f.split()) <= 60 and not f.endswith("?") and f[-1] in ".!":
            assunto = _assunto(f)
            if assunto:
                if assunto.split()[0].lower() in _ARTIGOS:
                    assunto = assunto[0].lower() + assunto[1:]
                adicionar(f"O que o material informa sobre {assunto}?", f)
    return pares[:n]


def dialogo_por_regras(pares, n_trocas):
    if len(pares) < 2:
        return []
    turnos = [pares[0]]
    for pergunta, resposta in pares[1:n_trocas]:
        turnos.append(("Entendi. " + pergunta, resposta))
    return turnos


def _registro_qa(pergunta, resposta, fonte, origem):
    return {"instruction": pergunta.strip(), "input": "", "output": resposta.strip(), "fonte": fonte,
            "_origem": origem, "_revisado": False}


def _registro_chat(turnos, fonte, origem, recusa=False):
    mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
    for usuario, assistente in turnos:
        mensagens.append({"role": "user", "content": usuario.strip()})
        mensagens.append({"role": "assistant", "content": assistente.strip()})
    return {"messages": mensagens, "fonte": fonte, "_origem": origem, "_revisado": False, "_recusa": recusa}


def gerar_qa_e_chat(trechos, cliente=None, progresso=None):
    """Etapa 7: percorre os trechos e devolve (qa, chat, registro da geração)."""
    qa, chat = [], []
    log = Counter()
    usar_llm, falhas_seguidas = cliente is not None, 0
    for i, t in enumerate(trechos):
        if progresso:
            progresso(i / max(1, len(trechos)), f"Gerando perguntas e diálogos ({i + 1}/{len(trechos)})")
        pares, turnos, origem, respondeu = [], [], "regras", False
        if usar_llm:
            prompt = PROMPT_GERACAO.format(tema=TEMA, n_qa=QA_POR_TRECHO, n_trocas=TROCAS_POR_DIALOGO,
                                           fonte=t["fonte"], trecho=t["text"])
            dados = _ler_json(chamar_llm(cliente, prompt))
            if isinstance(dados, dict):
                respondeu, falhas_seguidas, origem = True, 0, "llm"
                for p in dados.get("qa") or []:
                    perg, resp = str(p.get("pergunta", "")), str(p.get("resposta", ""))
                    if perg and resp and fidelidade(resp, t["text"]) >= LIMIAR_FIDELIDADE:
                        pares.append((perg, resp))
                    else:
                        log["descartados_por_fidelidade"] += 1
                turnos = [(str(d.get("usuario", "")), str(d.get("assistente", "")))
                          for d in (dados.get("dialogo") or []) if d.get("usuario") and d.get("assistente")]
                if any(fidelidade(a, t["text"]) < LIMIAR_FIDELIDADE for _, a in turnos):
                    log["descartados_por_fidelidade"] += 1
                    turnos = []
                time.sleep(PAUSA_ENTRE_CHAMADAS)
            else:
                falhas_seguidas += 1
                log["falhas_do_llm"] += 1
                if falhas_seguidas >= 3:
                    usar_llm = False
                    print("LLM desativado após 3 falhas seguidas; o restante será gerado por regras.")
                    print("Último erro da API:", ULTIMO_ERRO_LLM)
        if not respondeu:                                  # sem LLM (ou falhou): regras
            pares = qa_por_regras(t["text"], QA_POR_TRECHO)
            turnos = dialogo_por_regras(pares, TROCAS_POR_DIALOGO)
        qa += [_registro_qa(p, r, t["fonte"], origem) for p, r in pares]
        if len(turnos) >= 2:
            chat.append(_registro_chat(turnos, t["fonte"], origem))
        log[f"trechos_via_{origem}"] += 1
    return qa, chat, log

### Teste da chave
Rode esta célula antes de gerar os corpus. Ela diz se a chave foi encontrada e se o modelo responde.
No relatório final, a linha **Origem dos pares de Q&A** precisa mostrar `llm`; se mostrar `regras`, a IA não foi usada.

In [30]:
LLM_OK = testar_llm(ler_chave())

  Cota DIÁRIA da API esgotada. Tente amanhã ou use outra chave.
  LLM falhou (tentativa 1/4): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand  | aguardando 65s
  LLM falhou (tentativa 2/4): 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand  | aguardando 65s
Gemini funcionando com o modelo 'gemini-flash-latest'. Resposta de teste: ok


### Limites do assistente (recusas e encaminhamentos)
Exemplos escritos manualmente. Ensinam o modelo a **não responder fora do tema** e a **encaminhar para um profissional**
quando a pergunta pede diagnóstico, remédio, orientação jurídica ou dados pessoais. Adapte os textos se mudar o tema.

In [31]:
FONTE_RECUSAS = "exemplos_manuais_limites_do_assistente"

RECUSAS = [
    # ----- fora do tema -----
    [("Como faço para parcelar o IPTU atrasado?",
      "Desculpe, não consigo ajudar com esse assunto. Eu atendo apenas dúvidas sobre dengue e vacinação no SUS. Para falar de IPTU, procure a prefeitura da sua cidade."),
     ("Mas você não tem nem o telefone da prefeitura?",
      "Não tenho essa informação. O melhor caminho é o site ou o balcão de atendimento da prefeitura. Se tiver alguma dúvida sobre dengue ou vacinas, posso ajudar.")],
    [("Quem ganhou o jogo de ontem?",
      "Não tenho informações sobre futebol. Eu só respondo dúvidas sobre dengue e vacinação no SUS."),
     ("Ah, que pena. E sobre o que eu posso perguntar então?",
      "Você pode perguntar, por exemplo, como evitar o mosquito da dengue, quais são os sinais da doença ou onde se vacinar.")],
    [("Em quem eu devo votar na próxima eleição?",
      "Não posso opinar sobre eleições nem sobre candidatos. Meu atendimento é só sobre dengue e vacinação no SUS."),
     ("Mas qual candidato cuida melhor da saúde?",
      "Também não posso avaliar candidatos. Para conhecer as propostas, consulte os canais oficiais da Justiça Eleitoral. Se quiser, tiro dúvidas sobre dengue ou vacinas.")],
    [("Vale a pena colocar meu dinheiro na poupança?",
      "Não consigo orientar sobre investimentos. Eu atendo apenas dúvidas sobre dengue e vacinação no SUS. Para esse assunto, procure o seu banco ou um profissional da área."),
     ("Entendi. Então me ajuda com outra coisa?",
      "Claro, desde que seja sobre dengue ou vacinação. É só perguntar.")],
    [("Como eu faço para me inscrever no Bolsa Família?",
      "Esse assunto não faz parte do meu atendimento, que é só sobre dengue e vacinação no SUS. Para benefícios sociais, procure o CRAS mais perto da sua casa."),
     ("O CRAS também aplica vacina?",
      "Não. As vacinas do SUS são aplicadas nas Unidades Básicas de Saúde (UBS). No CRAS você resolve os assuntos de assistência social.")],
    [("Você pode escrever a redação da escola do meu filho?",
      "Não posso ajudar com trabalhos escolares. Meu atendimento é apenas sobre dengue e vacinação no SUS."),
     ("Nem se a redação for sobre dengue?",
      "Não escrevo a redação, mas posso explicar o que os materiais oficiais dizem sobre a dengue para ele estudar e escrever com as próprias palavras.")],
    # ----- encaminhamento para profissional -----
    [("Estou com febre e dor no corpo. Qual remédio eu tomo e quantas gotas?",
      "Não posso indicar remédios nem doses. Só um profissional de saúde pode avaliar o seu caso. Procure a Unidade Básica de Saúde (UBS) mais perto de você e evite tomar remédio por conta própria."),
     ("Mas é só uma febre, não dá para me dizer?",
      "Entendo a sua preocupação, mas a febre pode ter várias causas e o tratamento certo depende de uma avaliação. Vá à UBS. Se sentir dor forte na barriga, vômitos que não param ou sangramento, procure um serviço de urgência.")],
    [("Apareceram manchas vermelhas na minha pele. É dengue?",
      "Não consigo dizer se é dengue. Só um profissional de saúde, com exame e avaliação, pode confirmar. Procure a UBS mais próxima."),
     ("E se eu esperar uns dias para ver se passa?",
      "O mais seguro é não esperar. Procure a UBS para ser avaliado e, se piorar, vá a um serviço de urgência.")],
    [("Meu exame de sangue deu plaquetas baixas. Isso é grave?",
      "Não posso interpretar resultados de exames. Leve o exame a um médico ou à UBS para que um profissional avalie o seu caso."),
     ("Posso mandar a foto do exame para você olhar?",
      "Não precisa enviar, porque eu não analiso exames. Quem pode fazer isso é o profissional de saúde que acompanha você.")],
    [("Tenho uma doença crônica e tomo vários remédios. Posso tomar todas as vacinas?",
      "Essa decisão depende de uma avaliação individual, e eu não posso fazê-la. Converse com o profissional de saúde da UBS antes de se vacinar."),
     ("O que eu levo nessa consulta?",
      "Leve a sua caderneta de vacinação e a lista dos remédios que você usa, para o profissional poder orientar você com segurança.")],
    [("Meu filho pequeno está com febre alta há três dias e muito molinho. O que eu faço?",
      "Procure atendimento de saúde agora, na UBS ou em um serviço de urgência. Eu não posso avaliar o caso, e uma criança nessa situação precisa ser examinada por um profissional."),
     ("E se eu não conseguir levar ele?",
      "Em caso de emergência, ligue para o SAMU no 192. Não espere a criança piorar para buscar ajuda.")],
    [("O posto não tinha a vacina que eu precisava. Posso processar a prefeitura?",
      "Não posso dar orientação jurídica. Para saber sobre os seus direitos, procure a Defensoria Pública ou um advogado."),
     ("Tem algum lugar para eu reclamar?",
      "Você pode registrar a reclamação na ouvidoria do SUS, pelo Disque Saúde 136, e também na secretaria de saúde do seu município.")],
    [("Peguei dengue e faltei ao trabalho. Meu patrão é obrigado a pagar esses dias?",
      "Essa é uma dúvida trabalhista, e eu não posso orientar sobre isso. Procure o sindicato da sua categoria, um advogado ou a Defensoria Pública."),
     ("E o atestado, quem me dá?",
      "O atestado é dado pelo profissional de saúde que atender você. Peça a ele durante a consulta.")],
    # ----- proteção de dados pessoais -----
    [("Me passa o telefone pessoal da enfermeira que me atendeu no posto?",
      "Não compartilho dados pessoais de profissionais nem de pacientes. Para falar com a equipe, entre em contato diretamente com a sua UBS."),
     ("E o endereço de um vizinho que está com dengue, você sabe?",
      "Não tenho e não informo dados de outras pessoas. Se você viu um possível foco do mosquito, avise a UBS ou a vigilância em saúde do seu município.")],
]


def exemplos_de_recusa():
    chat = [_registro_chat(turnos, FONTE_RECUSAS, "manual", recusa=True) for turnos in RECUSAS]
    qa = []
    for turnos in RECUSAS:
        r = _registro_qa(turnos[0][0], turnos[0][1], FONTE_RECUSAS, "manual")
        r["_recusa"] = True
        qa.append(r)
    return qa, chat

## 8. Deduplicação, gravação e relatório de qualidade

In [32]:
def _impressao_digital(texto: str) -> str:
    base = re.sub(r"[^\w]+", " ", texto.lower()).strip()
    return hashlib.md5(base.encode("utf-8")).hexdigest()


CHAVES_DEDUP = {
    "texto": lambda e: e["text"],
    "qa": lambda e: e["instruction"],
    "chat": lambda e: " ".join(m["content"] for m in e["messages"] if m["role"] == "user"),
}


def remover_duplicatas(exemplos, chave):
    vistos, unicos = set(), []
    for e in exemplos:
        h = _impressao_digital(chave(e))
        if h not in vistos:
            vistos.add(h)
            unicos.append(e)
    return unicos, len(exemplos) - len(unicos)


def _publico(exemplo):
    """Tira os campos internos (começam com '_') antes de gravar."""
    return {k: v for k, v in exemplo.items() if not k.startswith("_")}


ARQUIVOS = {"texto": "corpus_texto.jsonl", "qa": "corpus_qa.jsonl", "chat": "corpus_chat.jsonl"}


def salvar_corpus(corpus):
    caminhos = []
    for nome, arquivo in ARQUIVOS.items():
        caminho = PASTA_SAIDA / arquivo
        with open(caminho, "w", encoding="utf-8") as f:
            for e in corpus[nome]:
                f.write(json.dumps(_publico(e), ensure_ascii=False) + "\n")
        caminhos.append(str(caminho))
    return caminhos


def _texto_do_exemplo(nome, e):
    if nome == "texto":
        return e["text"]
    if nome == "qa":
        return e["instruction"] + " " + e["output"]
    return " ".join(m["content"] for m in e["messages"])


def calcular_estatisticas(estado):
    corpus, est = estado["corpus"], {}
    for nome in ARQUIVOS:
        tokens = [contar_tokens(_texto_do_exemplo(nome, e)) for e in corpus[nome]]
        est[nome] = {
            "exemplos": len(tokens),
            "media_tokens": round(sum(tokens) / len(tokens), 1) if tokens else 0,
            "min_tokens": min(tokens, default=0),
            "max_tokens": max(tokens, default=0),
            "duplicatas_removidas": estado["duplicatas"][nome],
            "minimo_exigido": MINIMOS[nome],
            "por_origem": dict(Counter(e.get("_origem", "documento") for e in corpus[nome])),
        }
    tk = [e["_tokens"] for e in corpus["texto"]]
    est["texto"]["dentro_da_faixa"] = sum(MIN_TOKENS <= t <= MAX_TOKENS for t in tk)
    est["texto"]["documentos_curtos"] = sum(1 for e in corpus["texto"]
                                            if e.get("_doc_curto") and e["_tokens"] < MIN_TOKENS)
    est["texto"]["fora_da_faixa"] = len(tk) - est["texto"]["dentro_da_faixa"]
    est["recusas"] = sum(1 for e in corpus["chat"] if e.get("_recusa"))
    est["trocas_minimas_ok"] = all(sum(m["role"] == "assistant" for m in e["messages"]) >= 2
                                   for e in corpus["chat"])
    gerados = [e for n in ("qa", "chat") for e in corpus[n] if e.get("_origem") == "llm"]
    est["gerados_por_ia"] = len(gerados)
    est["revisados"] = sum(1 for e in gerados if e.get("_revisado"))
    return est


def _ok(condicao):
    return "✅" if condicao else "⚠️"


def montar_relatorio(estado) -> str:
    est, corpus = estado["estatisticas"], estado["corpus"]
    rotulos = {"texto": "Texto puro", "qa": "Perguntas e respostas", "chat": "Conversacional"}
    L = [f"# Relatório de qualidade: {TEMA}", "",
         f"Tokenizador: {TOKENIZADOR}. Documentos: {len(estado['documentos'])}. "
         f"Páginas/blocos lidos: {estado['n_paginas']}.", "",
         "## Estatísticas dos corpus", "",
         "| Corpus | Arquivo | Exemplos | Mínimo | Média de tokens | Mín. e máx. | Duplicatas removidas |",
         "|---|---|---|---|---|---|---|"]
    for nome, arquivo in ARQUIVOS.items():
        e = est[nome]
        L.append(f"| {rotulos[nome]} | `{arquivo}` | {e['exemplos']} {_ok(e['exemplos'] >= e['minimo_exigido'])} | "
                 f"{e['minimo_exigido']} | {e['media_tokens']} | {e['min_tokens']} a {e['max_tokens']} | "
                 f"{e['duplicatas_removidas']} |")
    t = est["texto"]
    L += ["",
          f"- Trechos entre {MIN_TOKENS} e {MAX_TOKENS} tokens: {t['dentro_da_faixa']} de {t['exemplos']} "
          f"{_ok(t['dentro_da_faixa'] + t['documentos_curtos'] == t['exemplos'])}"
          + (f" ({t['documentos_curtos']} trecho(s) menor(es) porque o documento inteiro tem menos de "
             f"{MIN_TOKENS} tokens, como um texto colado curto)" if t["documentos_curtos"] else "")
          + (f" ({t['fora_da_faixa'] - t['documentos_curtos']} trecho(s) de fim de documento sem um fim de frase "
             f"que permitisse ficar na faixa)" if t["fora_da_faixa"] > t["documentos_curtos"] else ""),
          f"- Diálogos com 2 ou mais trocas: {_ok(est['trocas_minimas_ok'])}",
          f"- Exemplos de recusa ou encaminhamento no corpus conversacional: {est['recusas']} "
          f"{_ok(est['recusas'] >= MINIMOS['recusas'])} (mínimo {MINIMOS['recusas']})",
          f"- Origem dos pares de Q&A: {est['qa']['por_origem']}",
          f"- Origem dos diálogos: {est['chat']['por_origem']}"]
    if est["gerados_por_ia"]:
        pct = 100 * est["revisados"] / est["gerados_por_ia"]
        L.append(f"- Revisão humana dos exemplos gerados por IA: {est['revisados']} de {est['gerados_por_ia']} "
                 f"({pct:.0f}%) {_ok(pct >= 100 * FRACAO_REVISAO)} (mínimo {FRACAO_REVISAO:.0%})")
    L += ["", "## Limpeza (itens removidos ou corrigidos)", "",
          "| Fonte | Cabeçalhos/rodapés | Nº de página | Sumário | URLs | Caracteres estranhos | Linhas repetidas | "
          "Hifens unidos | Letras iniciais unidas | Parágrafos de créditos/referências |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for fonte, s in estado["limpeza"].items():
        L.append(f"| {fonte} | {s['cabecalhos_rodapes']} | {s['numeros_de_pagina']} | {s['linhas_de_sumario']} | "
                 f"{s['urls']} | {s['caracteres_estranhos']} | {s['linhas_repetidas']} | {s['hifens_unidos']} | "
                 f"{s['letras_iniciais_unidas']} | {s['paragrafos_editoriais']} |")
    if estado["letras_unidas"]:
        L += ["", "Letras iniciais unidas (conferir): " + ", ".join(estado["letras_unidas"][:20])]
    a = estado["anonimizacao"]
    L += ["", "## Anonimização (LGPD)", "",
          f"Mascarados nos documentos: {a['cpfs']} CPF, {a['emails']} e-mails, {a['telefones']} telefones, "
          f"{a['nomes']} nomes. Telefones de utilidade pública (0800, 136, 192) são mantidos.", "",
          "Exemplo de demonstração (dados fictícios, não entram nos corpus):", "",
          "**Antes:**", "```", estado["exemplo_ficticio"]["antes"], "```",
          "**Depois:**", "```", estado["exemplo_ficticio"]["depois"], "```", ""]
    if estado["exemplos_anonimizacao"]:
        L += ["Trechos reais dos documentos" + (" (os parágrafos de créditos, depois de mascarados, são removidos "
              "do corpus):" if REMOVER_CREDITOS_E_REFERENCIAS else ":"), ""]
    for ex in estado["exemplos_anonimizacao"][:2]:
        L += ["**Antes:**", "```", ex["antes"], "```", "**Depois:**", "```", ex["depois"], "```", ""]
    g = estado["log_geracao"]
    L += ["## Geração", "",
          f"- Trechos processados com LLM: {g.get('trechos_via_llm', 0)}; por regras: {g.get('trechos_via_regras', 0)}",
          f"- Exemplos descartados pelo filtro de fidelidade: {g.get('descartados_por_fidelidade', 0)}",
          f"- Falhas de chamada ao LLM: {g.get('falhas_do_llm', 0)}"]
    if g.get("falhas_do_llm"):
        L.append(f"- Último erro da API: `{ULTIMO_ERRO_LLM[:200]}`")
    if g.get("trechos_via_llm"):
        L += [f"- Modelo usado: `{MODELO_LLM}`", "", "Prompt usado:", "```", PROMPT_GERACAO, "```"]
    rng = random.Random(SEMENTE)
    L += ["", "## Cinco exemplos aleatórios de cada corpus"]
    for nome in ARQUIVOS:
        L += ["", f"### {rotulos[nome]}", "```json"]
        for e in rng.sample(corpus[nome], min(5, len(corpus[nome]))):
            L.append(json.dumps(_publico(e), ensure_ascii=False))
        L.append("```")
    if estado["avisos"]:
        L += ["", "## Avisos", ""] + [f"- {a}" for a in estado["avisos"]]
    return "\n".join(L)


def finalizar(estado):
    """Recalcula estatísticas, grava os .jsonl e o relatório. Chamada após o pipeline e após a revisão."""
    estado["estatisticas"] = calcular_estatisticas(estado)
    estado["arquivos"] = salvar_corpus(estado["corpus"])
    estado["relatorio"] = montar_relatorio(estado)
    (PASTA_SAIDA / "relatorio_qualidade.md").write_text(estado["relatorio"], encoding="utf-8")
    (PASTA_SAIDA / "estatisticas.json").write_text(
        json.dumps(estado["estatisticas"], ensure_ascii=False, indent=2), encoding="utf-8")
    estado["arquivos"] += [str(PASTA_SAIDA / "relatorio_qualidade.md"), str(PASTA_SAIDA / "estatisticas.json")]
    return estado

## 9. Pipeline completo
Uma função que executa todas as etapas em ordem. É ela que a interface chama.

In [33]:
def executar_pipeline(arquivos=None, texto_colado="", api_key="", usar_llm=True, progresso=None):
    def passo(fracao, mensagem):
        print(mensagem)
        if progresso:
            progresso(fracao, mensagem)

    passo(0.02, "1-2/8 Ingestão e extração")
    documentos, avisos = ingerir(arquivos, texto_colado)
    if not documentos:
        raise ValueError("Nenhum texto para processar. Envie arquivos PDF, TXT, DOCX ou HTML, ou cole um texto.")

    ficticio, _, _ = anonimizar(TEXTO_DEMO_ANONIMIZACAO)
    estado = {"documentos": list(documentos), "avisos": avisos, "limpeza": {}, "amostras_limpeza": [],
              "letras_unidas": [], "exemplo_ficticio": {"antes": TEXTO_DEMO_ANONIMIZACAO, "depois": ficticio},
              "anonimizacao": Counter(), "exemplos_anonimizacao": [], "origem": tabela_origem(documentos),
              "n_paginas": sum(len(b) for b in documentos.values())}

    trechos = []
    for i, (fonte, blocos) in enumerate(documentos.items()):
        passo(0.05 + 0.25 * i / len(documentos), f"3-6/8 Limpeza, normalização, anonimização e segmentação: {fonte}")
        limpo, stats, unidas = limpar_documento(blocos)             # etapa 3
        normalizado = normalizar(limpo)                             # etapa 4
        anonimo, contagem, exemplos = anonimizar(normalizado)       # etapa 5
        if REMOVER_CREDITOS_E_REFERENCIAS:                          # etapa 3 (segunda parte)
            anonimo, removidos = remover_creditos_e_referencias(anonimo)
            stats["paragrafos_editoriais"] = len(removidos)
        partes = segmentar(anonimo, fonte)                          # etapa 6
        estado["letras_unidas"] += unidas
        estado["limpeza"][fonte] = stats
        estado["anonimizacao"] += contagem
        estado["exemplos_anonimizacao"] += exemplos
        estado["amostras_limpeza"].append({"fonte": fonte, "antes": blocos[0]["texto"][:700],
                                           "depois": normalizado[:700]})
        trechos += partes
    for chave in ("cpfs", "emails", "telefones", "nomes"):
        estado["anonimizacao"].setdefault(chave, 0)

    passo(0.32, "7/8 Geração dos corpus")
    cliente = criar_cliente_llm(api_key) if usar_llm else None
    if usar_llm and cliente is None:
        avisos.append("Sem chave de API válida: perguntas e diálogos foram gerados por regras"
                      + (f" ({MOTIVO_CHAVE})." if MOTIVO_CHAVE and not api_key else "."))
    faixa = (lambda f, m: progresso(0.32 + 0.6 * f, m)) if progresso else None
    qa, chat, log = gerar_qa_e_chat(trechos, cliente, faixa)
    qa_recusa, chat_recusa = exemplos_de_recusa()

    passo(0.94, "8/8 Deduplicação e relatório de qualidade")
    estado["corpus"], estado["duplicatas"] = {}, {}
    for nome, exemplos in (("texto", trechos), ("qa", qa + qa_recusa), ("chat", chat + chat_recusa)):
        estado["corpus"][nome], estado["duplicatas"][nome] = remover_duplicatas(exemplos, CHAVES_DEDUP[nome])
    estado["log_geracao"] = log
    estado = finalizar(estado)
    for nome in ARQUIVOS:
        if estado["estatisticas"][nome]["exemplos"] < MINIMOS[nome]:
            avisos.append(f"Corpus '{nome}' abaixo do mínimo ({estado['estatisticas'][nome]['exemplos']} de "
                          f"{MINIMOS[nome]}). Acrescente documentos ou aumente QA_POR_TRECHO.")
    estado = finalizar(estado)
    passo(1.0, "Concluído. Arquivos em " + str(PASTA_SAIDA.resolve()))
    return estado

### Revisão humana (20% dos exemplos gerados por IA)
O enunciado exige que o grupo revise pelo menos 20% dos exemplos quando usa IA. As funções abaixo sorteiam
a amostra, aplicam as correções e gravam um registro (`registro_revisao.csv`) para citar no relatório.

In [34]:
COLUNAS_REVISAO = ["corpus", "indice", "pergunta", "resposta", "fonte", "manter"]


def sortear_revisao(estado):
    """Sorteia a amostra de revisão entre os exemplos gerados por IA (ou por regras, se não houve IA)."""
    rng = random.Random(SEMENTE)
    linhas = []
    for nome in ("qa", "chat"):
        itens = list(enumerate(estado["corpus"][nome]))
        origem = "llm" if any(e.get("_origem") == "llm" for _, e in itens) else "regras"
        total = sum(1 for _, e in itens if e.get("_origem") == origem)
        candidatos = [i for i, e in itens if e.get("_origem") == origem and not e.get("_revisado")]
        k = min(len(candidatos), math.ceil(FRACAO_REVISAO * total))
        for i in sorted(rng.sample(candidatos, k)):
            e = estado["corpus"][nome][i]
            if nome == "qa":
                pergunta, resposta = e["instruction"], e["output"]
            else:
                pergunta = " / ".join(m["content"] for m in e["messages"] if m["role"] == "user")
                resposta = " / ".join(m["content"] for m in e["messages"] if m["role"] == "assistant")
            linhas.append([nome, i, pergunta, resposta, e["fonte"], True])
    return pd.DataFrame(linhas, columns=COLUNAS_REVISAO)


def aplicar_revisao(estado, tabela):
    """Marca como revisados, aplica edições nos pares de Q&A e remove o que foi desmarcado em 'manter'."""
    tabela = pd.DataFrame(tabela, columns=COLUNAS_REVISAO)
    remover = {"qa": set(), "chat": set()}
    registro = []
    for _, linha in tabela.iterrows():
        nome, i = linha["corpus"], int(linha["indice"])
        if nome not in remover or i >= len(estado["corpus"][nome]):
            continue
        e = estado["corpus"][nome][i]
        manter = str(linha["manter"]).strip().lower() in ("true", "1", "sim")
        acao = "aprovado"
        if not manter:
            remover[nome].add(i)
            acao = "removido"
        elif nome == "qa" and (linha["pergunta"] != e["instruction"] or linha["resposta"] != e["output"]):
            e["instruction"], e["output"] = str(linha["pergunta"]), str(linha["resposta"])
            acao = "editado"
        e["_revisado"] = True
        registro.append({"corpus": nome, "indice": i, "fonte": e["fonte"], "acao": acao})
    for nome, indices in remover.items():
        estado["corpus"][nome] = [e for i, e in enumerate(estado["corpus"][nome]) if i not in indices]
    pd.DataFrame(registro).to_csv(PASTA_SAIDA / "registro_revisao.csv", index=False, encoding="utf-8")
    estado = finalizar(estado)
    estado["arquivos"].append(str(PASTA_SAIDA / "registro_revisao.csv"))
    resumo = Counter(r["acao"] for r in registro)
    return estado, f"Revisão aplicada: {dict(resumo)}. Registro salvo em registro_revisao.csv."

## 10. Documentos de entrada sugeridos (opcional)
Fontes oficiais sobre dengue e vacinação. A célula tenta baixá-las; se algum link falhar (os sites do governo
às vezes bloqueiam o Colab), baixe o arquivo no seu computador e envie pela interface.
**Confira cada link e a contagem de páginas antes de entregar** e cite as fontes no relatório.

In [35]:
DOCUMENTOS_SUGERIDOS = {
    "dengue_saude_de_a_a_z.html":
        "https://www.gov.br/saude/pt-br/assuntos/saude-de-a-a-z/d/dengue",
    "vacinacao_ministerio_da_saude.html":
        "https://www.gov.br/saude/pt-br/vacinacao",
    "cartilha_vacinacao_agente_comunitario.pdf":
        "https://www.gov.br/saude/pt-br/acesso-a-informacao/participacao-social/consultas-publicas/2023/"
        "consulta-publica-cartilha-de-acompanhamento-da-vacinacao-pelo-agente-comunitario-de-saude/cartilhadoacs.pdf",
    "agente_comunitario_controle_da_dengue.pdf":
        "https://mosquito.saude.es.gov.br/Media/dengue/Arquivos/cartilha_acs_dengue_web.pdf",
    # o PDF do calendário da criança é uma imagem (sem texto para extrair); usamos a página em HTML
    "calendario_de_vacinacao.html":
        "https://www.gov.br/saude/pt-br/vacinacao/calendario",
}
PASTA_ENTRADA = Path("documentos_entrada")


def baixar_documentos(documentos=DOCUMENTOS_SUGERIDOS):
    import requests
    PASTA_ENTRADA.mkdir(exist_ok=True)
    baixados = []
    for nome, url in documentos.items():
        destino = PASTA_ENTRADA / nome
        try:
            if not destino.exists():
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                destino.write_bytes(r.content)
            baixados.append(str(destino))
            print("OK     ", nome)
        except Exception as erro:
            print("FALHOU ", nome, "->", str(erro)[:80])
    return baixados

In [36]:
arquivos_baixados = baixar_documentos()

OK      dengue_saude_de_a_a_z.html
OK      vacinacao_ministerio_da_saude.html
OK      cartilha_vacinacao_agente_comunitario.pdf
OK      agente_comunitario_controle_da_dengue.pdf
OK      calendario_de_vacinacao.html


## 11. Interface Gradio
Envie os documentos, cole um texto se quiser e clique em **Executar pipeline**.
Os corpus aparecem para download na aba 5 e também ficam gravados na pasta `saida/` do Colab (são os mesmos arquivos).
Os documentos baixados na seção 10 ficam em `documentos_entrada/`: baixe-os de lá para o computador se quiser enviá-los aqui.
Cada aba mostra o resultado de uma etapa; a última permite revisar a amostra de 20%.

In [37]:
import gradio as gr


def _status_da_chave():
    if ler_chave():
        return f"Chave do Gemini encontrada. Modelo: `{MODELO_LLM}`."
    return ("**Chave do Gemini não encontrada**" + (f": {MOTIVO_CHAVE}" if MOTIVO_CHAVE else "") +
            ". Cole a chave no campo acima ou corrija o secret e rode esta célula de novo. "
            "Sem chave, as perguntas saem por regras.")


def _md_limpeza(estado):
    L = ["### Antes e depois da limpeza e normalização (início de cada documento)"]
    for a in estado["amostras_limpeza"]:
        L += [f"\n**{a['fonte']}**", "\nAntes:", "```", a["antes"], "```", "Depois:", "```", a["depois"], "```"]
    return "\n".join(L)


def _md_anonimizacao(estado):
    a = estado["anonimizacao"]
    L = [f"### Dados pessoais mascarados\n\nCPF: **{a['cpfs']}** · e-mails: **{a['emails']}** · "
         f"telefones: **{a['telefones']}** · nomes: **{a['nomes']}**\n"]
    for ex in [estado["exemplo_ficticio"]] + estado["exemplos_anonimizacao"]:
        L += ["Antes:", "```", ex["antes"], "```", "Depois:", "```", ex["depois"], "```"]
    return "\n".join(L)


def _amostra(estado, nome, n=3):
    return "\n".join(json.dumps(_publico(e), ensure_ascii=False, indent=2) for e in estado["corpus"][nome][:n])


def ui_executar(arquivos, texto_colado, api_key, usar_llm, progress=gr.Progress()):
    try:
        estado = executar_pipeline(arquivos, texto_colado, (api_key or "").strip(), usar_llm, progress)
    except Exception as erro:
        raise gr.Error(str(erro))
    return (estado, estado["origem"], _md_limpeza(estado), _md_anonimizacao(estado),
            _amostra(estado, "texto"), _amostra(estado, "qa"), _amostra(estado, "chat"),
            estado["arquivos"], estado["relatorio"], sortear_revisao(estado), "")


def ui_revisar(estado, tabela):
    if not estado:
        raise gr.Error("Execute o pipeline antes de revisar.")
    estado, mensagem = aplicar_revisao(estado, tabela)
    # devolve uma nova amostra (ainda não revisada) para quem quiser revisar mais de 20%
    return estado, mensagem, estado["relatorio"], estado["arquivos"], sortear_revisao(estado)


with gr.Blocks(title="Fábrica de Corpus CitizenTech") as app:
    gr.Markdown(f"# Fábrica de Corpus CitizenTech\nTema: **{TEMA}**. Documentos públicos entram, três corpus `.jsonl` saem.")
    estado_ui = gr.State()
    with gr.Tab("1. Entrada"):
        with gr.Row():
            with gr.Column():
                entrada_arquivos = gr.File(label="Documentos (PDF, TXT, DOCX, HTML)", file_count="multiple",
                                           file_types=FORMATOS_ACEITOS)
                entrada_texto = gr.Textbox(label="Ou cole um texto aqui", lines=6,
                                           placeholder="Texto colado entra no pipeline com a fonte 'texto_colado'.")
            with gr.Column():
                entrada_chave = gr.Textbox(label="Chave da API Gemini (opcional)", type="password",
                                           value=ler_chave())
                gr.Markdown(_status_da_chave())
                entrada_llm = gr.Checkbox(label="Gerar perguntas e diálogos com IA (sem chave, usa regras)", value=True)
                botao = gr.Button("Executar pipeline", variant="primary")
                gr.Markdown("Requisito do enunciado: 3 a 5 documentos, 20 páginas ou mais no total.")
    with gr.Tab("2. Extração e origem"):
        saida_origem = gr.Dataframe(label="De onde veio cada bloco de texto", wrap=True)
    with gr.Tab("3. Limpeza e normalização"):
        saida_limpeza = gr.Markdown()
    with gr.Tab("4. Anonimização"):
        saida_anon = gr.Markdown()
    with gr.Tab("5. Corpus"):
        saida_arquivos = gr.File(label="Arquivos gerados (clique para baixar)", file_count="multiple")
        saida_texto = gr.Code(label="corpus_texto.jsonl (3 primeiros)", language="json")
        saida_qa = gr.Code(label="corpus_qa.jsonl (3 primeiros)", language="json")
        saida_chat = gr.Code(label="corpus_chat.jsonl (3 primeiros)", language="json")
    with gr.Tab("6. Relatório de qualidade"):
        saida_relatorio = gr.Markdown()
    with gr.Tab("7. Revisão humana"):
        gr.Markdown("Amostra de 20% dos exemplos gerados. Edite pergunta e resposta dos pares de Q&A se precisar, "
                    "desmarque **manter** para excluir um exemplo e clique em **Aplicar revisão**. "
                    "Depois de aplicar, a tabela traz uma nova amostra, caso o grupo queira revisar mais.")
        tabela_revisao = gr.Dataframe(headers=COLUNAS_REVISAO, interactive=True, wrap=True,
                                      datatype=["str", "number", "str", "str", "str", "bool"])
        botao_revisao = gr.Button("Aplicar revisão")
        mensagem_revisao = gr.Markdown()

    botao.click(ui_executar, [entrada_arquivos, entrada_texto, entrada_chave, entrada_llm],
                [estado_ui, saida_origem, saida_limpeza, saida_anon, saida_texto, saida_qa, saida_chat,
                 saida_arquivos, saida_relatorio, tabela_revisao, mensagem_revisao])
    botao_revisao.click(ui_revisar, [estado_ui, tabela_revisao],
                        [estado_ui, mensagem_revisao, saida_relatorio, saida_arquivos, tabela_revisao])

In [42]:
app.launch(share=True, debug=False)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f962ed763bea8d0b3a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Execução sem interface (opcional)
Alternativa à interface: gera os corpus direto pelas células, usando os documentos baixados na seção 10.
Vem desligada para não gastar a cota da API duas vezes. Para usar, troque `RODAR_SEM_INTERFACE` para `True`.
Os arquivos saem na mesma pasta `saida/` que a interface usa.

In [44]:
RODAR_SEM_INTERFACE = False

arquivos_locais = sorted(str(p) for p in PASTA_ENTRADA.glob("*") if p.suffix.lower() in FORMATOS_ACEITOS)
if not RODAR_SEM_INTERFACE:
    print("Execução sem interface desligada. Use a interface Gradio acima.")
elif arquivos_locais:
    estado = executar_pipeline(arquivos_locais, api_key=ler_chave())
    print(estado["relatorio"])
else:
    print("Nenhum arquivo em documentos_entrada/. Use a interface acima ou envie os arquivos para essa pasta.")

Execução sem interface desligada. Use a interface Gradio acima.


In [43]:
from IPython.display import Markdown
from pathlib import Path

relatorio = Path("saida/relatorio_qualidade.md")
if relatorio.exists():
    display(Markdown(relatorio.read_text(encoding="utf-8")))
else:
    print("Relatório ainda não gerado. Use a interface Gradio acima, clique em "
          "'Executar pipeline' e rode esta célula de novo.")

# Relatório de qualidade: Dengue e vacinação no SUS

Tokenizador: tiktoken (cl100k_base). Documentos: 5. Páginas/blocos lidos: 41.

## Estatísticas dos corpus

| Corpus | Arquivo | Exemplos | Mínimo | Média de tokens | Mín. e máx. | Duplicatas removidas |
|---|---|---|---|---|---|---|
| Texto puro | `corpus_texto.jsonl` | 77 ✅ | 50 | 303.8 | 261 a 496 | 0 |
| Perguntas e respostas | `corpus_qa.jsonl` | 182 ✅ | 100 | 57.2 | 25 a 157 | 29 |
| Conversacional | `corpus_chat.jsonl` | 82 ✅ | 30 | 202.9 | 142 a 326 | 1 |

- Trechos entre 256 e 512 tokens: 77 de 77 ✅
- Diálogos com 2 ou mais trocas: ✅
- Exemplos de recusa ou encaminhamento no corpus conversacional: 14 ✅ (mínimo 10)
- Origem dos pares de Q&A: {'regras': 168, 'manual': 14}
- Origem dos diálogos: {'regras': 68, 'manual': 14}

## Limpeza (itens removidos ou corrigidos)

| Fonte | Cabeçalhos/rodapés | Nº de página | Sumário | URLs | Caracteres estranhos | Linhas repetidas | Hifens unidos | Letras iniciais unidas | Parágrafos de créditos/referências |
|---|---|---|---|---|---|---|---|---|---|
| cartilha_vacinacao_agente_comunitario.docx | 0 | 0 | 0 | 9 | 85 | 0 | 0 | 15 | 11 |
| dengue_saude_de_a_a_z.html | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 1 |
| calendario_de_vacinacao.html | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 1 |
| agente_comunitario_controle_da_dengue.pdf | 43 | 12 | 0 | 8 | 23 | 0 | 96 | 0 | 12 |
| vacinacao_ministerio_da_saude.txt | 0 | 0 | 0 | 0 | 1 | 1 | 0 | 0 | 1 |

Letras iniciais unidas (conferir): As, Quais, Como, Como, Conhecendo, Desenvolvendo, Desenvolvendo, Realizando, Realizando, Quando, Realizando, Mobilizando, Combatendo, Bibliografia, Anexos

## Anonimização (LGPD)

Mascarados nos documentos: 0 CPF, 1 e-mails, 2 telefones, 20 nomes. Telefones de utilidade pública (0800, 136, 192) são mantidos.

Exemplo de demonstração (dados fictícios, não entram nos corpus):

**Antes:**
```
Relato de atendimento (dados fictícios). A paciente Maria Aparecida de Souza, CPF 123.456.789-09, procurou a UBS com febre e dor no corpo. Ela foi atendida pela Dra. Helena Martins e orientada a beber bastante água. O retorno foi marcado pelo telefone (11) 91234-5678 e a confirmação enviada para maria.souza@exemplo.com. Em caso de dúvida, a população pode ligar para o Disque Saúde 136.
```
**Depois:**
```
Relato de atendimento (dados fictícios). A paciente [NOME], CPF [CPF], procurou a UBS com febre e dor no corpo. Ela foi atendida pela Dra. [NOME] e orientada a beber bastante água. O retorno foi marcado pelo telefone [TELEFONE] e a confirmação enviada para [EMAIL]. Em caso de dúvida, a população pode ligar para o Disque Saúde 136.
```

Trechos reais dos documentos (os parágrafos de créditos, depois de mascarados, são removidos do corpus):

**Antes:**
```
s da estratégia saúde da família: um dispositivo indispensável para o planejamento local. Texto Contexto Enfermagem, v. 28, e20170477, 2019. Disponível em:. Acesso em: 21 mar. 2023.
```
**Depois:**
```
s da estratégia saúde da família: um dispositivo indispensável para o planejamento local. [NOME] Enfermagem, v. 28, e20170477, 2019. Disponível em:. Acesso em: 21 mar. 2023.
```

**Antes:**
```
Compartilhe: Compartilhe por Facebook Compartilhe por LinkedIn Compartilhe por WhatsApp link para Copiar para área de transferência
```
**Depois:**
```
Compartilhe: Compartilhe por [NOME] por LinkedIn Compartilhe por WhatsApp link para Copiar para área de transferência
```

## Geração

- Trechos processados com LLM: 0; por regras: 77
- Exemplos descartados pelo filtro de fidelidade: 0
- Falhas de chamada ao LLM: 0

## Cinco exemplos aleatórios de cada corpus

### Texto puro
```json
{"text": "sociais; Instituições de longa permanência para idosos (ILPI); Unidades socioeducativas; Unidades do sistema prisional vinculadas às equipes da Atenção Primária à Saúde; Shoppings; Supermercados;\n- Terminais rodoviários de grande circulação; Entre outros serviços existentes na microárea de atuação.\nCombatendo as Fake News\nFake news são notícias falsas que podem enganar as pessoas, causar insegurança e medo de se vacinar.\nComo fazer para identificar fake news? Observar os conteúdos apelativos e sensacionalistas; Pesquisar em fontes seguras e oficiais a veracidade da notícia; Suspeitar de notícias alarmantes e sem autoria; Desconfiar de sites desconhecidos; Observar erros gramaticais nas notícias; Suspeitar de pedidos de compartilhamentos.\nQual é o resultado das fake news? Hesitação vacinal (dúvida ou recusa de se vacinar, apesar da disponibilidade da vacina); Diminuição da cobertura vacinal; Desproteção da população; Aumento de casos novos das doenças imunopreveníveis; Risco de reintrodução/ retorno de doenças eliminadas; Óbitos por doenças preveníveis pela vacinação.", "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"text": "Mas, como a vacina com micro-organismo fraco ou morto vai se comportar no corpo?\nApós administrada a vacina, o corpo identificará o micro-organismo e descobrirá como combatê-lo.\nEsse combate se dá por indução do sistema imunológico, que fabricará anticorpos para defesa do indivíduo contra o micro-organismo, e a resposta gerará uma memória imunológica do vírus ou bactéria específicos.\nApós vacinado, caso o corpo seja novamente exposto ao microrganismo, ele estará preparado para combatê-lo antes de causar a doença.\nO que são doenças imunopreveníveis?\nA doença imunoprevenível é aquela que pode ser evitada após aplicar a vacina.\nQuais são as vacinas disponíveis no Calendário Nacional de Vacinação, do Sistema Único de Saúde (SUS), e de que doenças elas protegem?\nO Programa Nacional de Imunizações (PNI), do Ministério da Saúde, adquire vacinas para proteger toda a população brasileira.", "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"text": "Doenças evitadas: doenças meningocócicas causadas pelos sorogrupos A, C, W-135 e Y\nVacina tríplice viral SCR (1ª dose)\nDoenças evitadas: sarampo, caxumba, rubéola e síndrome da rubéola congênita.\n15 meses\nVacina DTP (1ª dose reforço)\nDoenças evitadas: difteria, tétano e coqueluche\nVacina poliomielite inativada VIP (1ª dose reforço)\nDoença evitada: poliomielite (paralisia infantil)\nVacina tríplice viral SCR (2ª dose)\nDoenças evitadas: sarampo, caxumba, rubéola, síndrome da rubéola congênita.\nVacina varicela (1ª dose)\nDoença evitada: varicela (catapora)\nObs.: Em casos de indisponibilidade da vacina varicela monovalente, a vacina tetraviral poderá ser utilizada.\nVacina hepatite A (1 dose)\nDoença evitada: hepatite A\n4 anos\nVacina DTP (2º dose reforço)", "fonte": "calendario_de_vacinacao.html"}
{"text": "Vacina poliomielite inativada VIP (2ª dose)\nDoença evitada: poliomielite (paralisia infantil)\nVacina rotavírus humano (2ª dose)\nDoença evitada: gastrenterite viral (diarreia e vômitos)\nObs.: Atenção aos prazos. A 1ª dose deve ser aplicada entre 1 mês e 15 dias e 11 meses e 29 dias. A 2ª dose, entre 3 meses e 15 dias e 23 meses e 29 dias. Intervalo de 60 dias entre as doses. Se a 1ª dose não for administrada no período indicado, a criança perderá a oportunidade de vacinação contra o rotavírus.\nVacina pneumocócica 10-valente (2ª dose)\nDoenças evitadas: doenças pneumocócicas invasivas\n5 meses\nVacina meningocócica C (2ª dose)\nDoenças evitadas: doenças meningocócicas causadas pelo sorogrupo C (meningite, encefalite, meningoencefalite)\n6 meses\nVacina penta (DTP+Hib+HB) (3ª dose)", "fonte": "calendario_de_vacinacao.html"}
{"text": "Portanto, em períodos fora da sazonalidade da doença é que ações preventivas devem ser adotadas. É o momento ideal para manutenção de medidas que visem impedir epidemias futuras. Nesse sentido, além das ações realizada pelos agentes de saúde, a população deve fazer a sua parte:\nUso de telas nas janelas e repelentes em áreas de reconhecida transmissão;\nRemoção de recipientes nos domicílios que possam se transformar em criadouros de mosquitos;\nVedação dos reservatórios e caixas de água;\nDesobstrução de calhas, lajes e ralos;\nParticipação na fiscalização das ações de prevenção e controle da dengue executadas pelo Sistema Único de Saúde (SUS).\nTratamento\nO tratamento é baseado principalmente na reposição de líquidos adequada. Por isso, conforme orientação médica, em casa deve-se realizar:\nRepouso;\nIngestão de líquidos;\nNão se automedicar e procurar imediatamente o serviço de urgência em caso de sangramentos ou surgimento de pelo menos um sinal de alarme;\nRetorno para reavaliação clínica conforme orientação médica.\nAinda não existe tratamento específico para a doença.\nVocê viu?\nAedes aegypti\nArboviroses\nMais informações\nAções de enfrentamento à dengue\nAções de comunicação e mobilização\nCOE Dengue e outras Arboviroses (2025)\nNotificações de casos suspeitos\nSala Nacional de Arboviroses\nSituação Epidemiológica\nNotas técnicas e informativas\nObservatório de Arboviroses\nPublicações\nPerguntas Frequentes\nOuvidoria-Geral do SUS\nTeleatendimento: de segunda-feira a sexta-feira, das 8h às 20h, e aos sábados, das 8h às 18h.\nMembro da Vaccine Safety Net (VSN)\nOrganização Mundial da Saúde – OMS\nO logotipo da VSN é de propriedade da OMS e utilizado com autorização.", "fonte": "dengue_saude_de_a_a_z.html"}
```

### Perguntas e respostas
```json
{"instruction": "O que o material informa sobre doenças imunopreveníveis?", "input": "", "output": "Utilizar termos usuais para falar sobre o que é ter saúde e como fazer para prevenir doenças imunopreveníveis; Debater sobre o que é verdade e o que é falso acerca das vacinas, de forma a esclarecer sobre as fake news.", "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"instruction": "O que é a visita domiciliar?", "input": "", "output": "A visita domiciliar é uma das principais atividades na sua microárea de trabalho.", "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"instruction": "Você pode escrever a redação da escola do meu filho?", "input": "", "output": "Não posso ajudar com trabalhos escolares. Meu atendimento é apenas sobre dengue e vacinação no SUS.", "fonte": "exemplos_manuais_limites_do_assistente"}
{"instruction": "O que o material informa sobre os ambientes saudáveis?", "input": "", "output": "Discuta com a comunidade as possibilidades de novos destinos para o lixo reciclável. essas medidas contribuem para evitar a reprodução do mosquito da dengue e para tornar os ambientes saudáveis. esta casa recebeu a visita da nossa ACS.", "fonte": "agente_comunitario_controle_da_dengue.pdf"}
{"instruction": "O que o material informa sobre cadastro individual?", "input": "", "output": "Veja as atividades que você pode realizar para conhecer o que existe na sua microárea de abrangência: Atividades de cadastro domiciliar e territorial; e Atividades de cadastro individual.", "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
```

### Conversacional
```json
{"messages": [{"role": "system", "content": "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a Unidade Básica de Saúde (UBS) ou o órgão responsável."}, {"role": "user", "content": "Apareceram manchas vermelhas na minha pele. É dengue?"}, {"role": "assistant", "content": "Não consigo dizer se é dengue. Só um profissional de saúde, com exame e avaliação, pode confirmar. Procure a UBS mais próxima."}, {"role": "user", "content": "E se eu esperar uns dias para ver se passa?"}, {"role": "assistant", "content": "O mais seguro é não esperar. Procure a UBS para ser avaliado e, se piorar, vá a um serviço de urgência."}], "fonte": "exemplos_manuais_limites_do_assistente"}
{"messages": [{"role": "system", "content": "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a Unidade Básica de Saúde (UBS) ou o órgão responsável."}, {"role": "user", "content": "O que o material informa sobre participação ativa?"}, {"role": "assistant", "content": "Importância da participação ativa de todos os setores da sociedade O controle da dengue exige um esforço de todos os profissionais de saúde, gestores e população."}, {"role": "user", "content": "Entendi. O que o material informa sobre Aedes aegypti?"}, {"role": "assistant", "content": "É importante lembrar que, para se reproduzir, o mosquito Aedes aegypti se utiliza de todo tipo de recipientes que as pessoas costumam usar nas atividades do dia-a-dia: garrafas e embalagens descartáveis, latas, pneus, plásticos, entre outros."}], "fonte": "agente_comunitario_controle_da_dengue.pdf"}
{"messages": [{"role": "system", "content": "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a Unidade Básica de Saúde (UBS) ou o órgão responsável."}, {"role": "user", "content": "E por que as altas coberturas vacinais são importantes no meu território?"}, {"role": "assistant", "content": "A vacina promove uma barreira de proteção contra doenças na comunidade ao diminuir a circulação dessas doenças, que se tornam cada vez mais raras, até desaparecerem da comunidade. Se as metas de vacinação não forem alcançadas no território, podem se formar \"bolsões\" de pessoas não vacinadas, deixando a comunidade exposta às doenças imunopreveníveis."}, {"role": "user", "content": "Entendi. Como você pode cooperar para o alcance das altas coberturas vacinais?"}, {"role": "assistant", "content": "Como membro da equipe da Atenção Primária à Saúde (APS), você tem papel fundamental na criação e no fortalecimento do vínculo da UBS com a comunidade, sendo o elo entre os profissionais de saúde e a população."}], "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"messages": [{"role": "system", "content": "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a Unidade Básica de Saúde (UBS) ou o órgão responsável."}, {"role": "user", "content": "O que o material informa sobre imunobiológico cuja proteção?"}, {"role": "assistant", "content": "As vacinas ofertadas na rotina dos serviços de saúde são definidas nos calendários de vacinação, nos quais estão estabelecidos: tipos de vacina; número de doses do esquema básico e dos reforços; idades (mínima e máxima) para a administração de cada dose; e intervalo entre uma dose e outra, no caso do imunobiológico cuja proteção exija mais de uma dose."}, {"role": "user", "content": "Entendi. O que o material informa sobre a população indígena?"}, {"role": "assistant", "content": "Importante: a população indígena também está incluída em todos esses grupos!"}], "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
{"messages": [{"role": "system", "content": "Você é um atendente de saúde pública do SUS. Responda de forma simples e educada, usando apenas informações oficiais sobre dengue e vacinação. Se a dúvida fugir do tema ou exigir avaliação de um profissional, explique isso e oriente a pessoa a procurar a Unidade Básica de Saúde (UBS) ou o órgão responsável."}, {"role": "user", "content": "Quando orientar a ida à sala de vacina?"}, {"role": "assistant", "content": "Sem vacinação ou não sabe Usuário não fez a vacina ou não sabe informar. Sem comprovante vacinal Sem comprovação vacinal por perda do documento, entre outros."}, {"role": "user", "content": "Entendi. O que o material informa sobre comprovação vacinal?"}, {"role": "assistant", "content": "Sem comprovante vacinal Sem comprovação vacinal por perda do documento, entre outros."}], "fonte": "cartilha_vacinacao_agente_comunitario.docx"}
```